# Iowa Nutrient Reduction Strategy — BMP Tracking Data Download

Downloads the Iowa Nutrient Reduction Strategy (INRS) tracking dataset published by
Iowa State University and filters it to the HUC-8 watershed-level BMP practice records
most useful for water quality modeling.

The full dataset covers 2003–2022 and tracks conservation practice adoption (bioreactors,
saturated buffers, CREP wetlands, cover crops, erosion control structures) alongside
water quality monitoring, farmer survey results, and funding/staffing inputs.
Only the `HUC8 Practice Adoption` rows are retained in the cleaned output — these
give cumulative or annual practice counts and acres by HUC-8 watershed and year,
which join directly to the NHDPlus HUC-12 boundaries already in the project.

**Source**: Iowa State University ArcGIS Online — INRS Tracking Dataset  
Item ID: `03cf4991b8bb46fd8764ebdaab80baf6`  
URL: `https://isugisf.maps.arcgis.com/sharing/rest/content/items/<item_id>/data`

**Outputs**
- `data/tabular/01_raw/bmp/iowa-nrs-tracking.csv` — full 20K-row dataset (all indicators)
- `data/tabular/01_raw/bmp/iowa-nrs-bmp-huc8.csv` — 2,195 HUC-8 BMP practice rows

In [ ]:
import io
import urllib.request

import pandas as pd
from pathlib import Path

RAW_DIR = Path('../../data/tabular/01_raw/bmp')
RAW_DIR.mkdir(parents=True, exist_ok=True)

ITEM_ID = '03cf4991b8bb46fd8764ebdaab80baf6'
URL = f'https://isugisf.maps.arcgis.com/sharing/rest/content/items/{ITEM_ID}/data'

# Categories to keep in the HUC-8 BMP output → canonical practice label
PRACTICE_MAP = {
    'CREP Wetlands':                                               'crep_wetland',
    'Bioreactors and Saturated Buffers':                           'bioreactor_sat_buffer',
    'Bioreactors and Saturated Buffers (Updated 2022)':            'bioreactor_sat_buffer',
    'Bioreactors & Saturated Buffers (updated 2022)':              'bioreactor_sat_buffer',
    'Bioreactor':                                                  'bioreactor',
    'Bioreactor (updated 2022)':                                   'bioreactor',
    'Saturated Buffer':                                            'sat_buffer',
    'Saturated Buffer (updated 2022)':                             'sat_buffer',
    'Multi-Purpose Oxbow':                                         'oxbow',
    'Bioreactors, Saturated Buffers, and Multi-Purpose Oxbow':     'bioreactor_sat_buffer',
    'Nitrate Removal Wetland':                                     'nitrate_wetland',
    'Nitrate Removal Wetland (updated 2022 INRS Tracking period)': 'nitrate_wetland',
    'Water Quality Wetlands':                                      'water_quality_wetland',
    'Erosion Control':                                             'erosion_control',
    '340':                                                         'cover_crop',   # NRCS practice code
    'Cover Crops':                                                 'cover_crop',
    'Cover Crop':                                                  'cover_crop',
}

## 1. Download and save full dataset

In [ ]:
print('Downloading Iowa NRS Tracking CSV...', flush=True)
with urllib.request.urlopen(URL, timeout=60) as r:
    raw = r.read()

df = pd.read_csv(io.BytesIO(raw), encoding='latin-1', dtype=str)
print(f'Downloaded: {len(df):,} rows, {len(df.columns)} columns')

out_full = RAW_DIR / 'iowa-nrs-tracking.csv'
df.to_csv(out_full, index=False, encoding='utf-8')
print(f'Saved → {out_full}')

print('\nmeasurableIndicator breakdown:')
print(df['measurableIndicator'].value_counts().to_string())

## 2. Filter to HUC-8 BMP practice rows

In [ ]:
bmp = df[
    df['category'].isin(PRACTICE_MAP.keys()) &
    df['huc8'].notna() &
    df['unit'].isin(['Number', 'Acres', 'acres'])
].copy()

bmp['practice_type'] = bmp['category'].map(PRACTICE_MAP)

bmp_clean = (
    bmp[['year', 'practice_type', 'category', 'assessment', 'value', 'unit', 'huc8', 'huc8Name']]
    .rename(columns={'huc8': 'huc8_code', 'huc8Name': 'huc8_name'})
    .sort_values(['practice_type', 'huc8_code', 'year'])
    .reset_index(drop=True)
)

out_bmp = RAW_DIR / 'iowa-nrs-bmp-huc8.csv'
bmp_clean.to_csv(out_bmp, index=False)
print(f'Saved {len(bmp_clean):,} BMP rows → {out_bmp}')
bmp_clean.head()

## 3. Summary

In [ ]:
print('HUC-8 BMP rows:          ', len(bmp_clean))
print('HUC-8 watersheds covered:', bmp_clean['huc8_code'].nunique())
print('Year range:              ', bmp_clean['year'].min(), '–', bmp_clean['year'].max())

print('\nPractice type x unit breakdown:')
print(bmp_clean.groupby(['practice_type', 'unit']).size().to_string())

print('\nJoin note:')
print('  huc8_code (8-digit) joins to the first 8 digits of HUC-12 codes')
print('  in data/spatial/nhdplus/wbd-huc12-iowa/WBDSnapshot_Iowa.shp')